In [102]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv 
import os

load_dotenv("../rock/.env")                       #Cargamos nuestras contraseñas en el .env
SQL_pass = os.getenv("SQL")

In [103]:
def conectar_db():
    # Ajusta tus credenciales
    usuario = 'root'
    password = SQL_pass
    host = "127.0.0.1"
    db = 'music_db' # El nombre que pusiste en el CREATE SCHEMA
    
    engine = create_engine(f'mysql+mysqlconnector://{usuario}:{password}@{host}/{db}')
    return engine

In [104]:
def subir_artistas(path_csv1, engine):
    df = pd.read_csv(path_csv1)
    
    # 1. Renombrar y limpiar (como antes)
    df = df.rename(columns={
        'id_artist': 'id_artist',
        'artist': 'artist_name',
        'bio': 'artist_bio',
        'listeners': 'artist_listeners',
        'playcount': 'artist_playcount',
        'similar_artist': 'similar_artist'
    })
    df['artist_name'] = df['artist_name'].str.strip()
    
    # 2. ELIMINAR DUPLICADOS DENTRO DEL PROPIO CSV
    df = df.drop_duplicates(subset=['id_artist'])
    
    # 3. COMPROBAR QUÉ ARTISTAS YA ESTÁN EN SQL (Para no repetir)
    try:
        artistas_en_db = pd.read_sql('SELECT id_artist FROM artists', con=engine)
        # Filtramos el dataframe: solo nos quedamos con los que NO están en la DB
        df = df[~df['id_artist'].isin(artistas_en_db['id_artist'])]
    except:
        # Si la tabla está vacía o no existe, seguimos con todo el DF
        pass

    if not df.empty:
        df.to_sql('artists', con=engine, if_exists='append', index=False)
        print(f"✅ Se han subido {len(df)} artistas nuevos.")
    else:
        print("ℹ️ No hay artistas nuevos para subir.")
    
    # Devolvemos todos los artistas que hay ahora en la DB para el cruce de tracks
    return pd.read_sql('SELECT * FROM artists', con=engine)

In [105]:
def procesar_y_subir_tracks(lista_csv_generos, df_artists, engine):
    mapeo_generos = {"rock": 1, "indie": 2, "latin": 3, "reggaeton": 4, "hip-hop": 5}

    for archivo in lista_csv_generos:
        df_tracks = pd.read_csv(archivo)
        
        # Mapeo y cruce (igual que antes)
        df_tracks['artist'] = df_tracks['artist'].str.strip()
        df_tracks['genre'] = df_tracks['genre'].str.strip().str.lower()
        df_unido = df_tracks.merge(df_artists[['id_artist', 'artist_name']], 
                                   left_on='artist', right_on='artist_name', how='left')
        df_unido['id_genre'] = df_unido['genre'].map(mapeo_generos)
        
        df_final = df_unido[['id_artist', 'id_genre', 'track_name', 'year']].dropna()

        # CONTROL DE DUPLICADOS PARA TRACKS
        # Leemos lo que ya hay en la tabla tracks
        try:
            tracks_existentes = pd.read_sql('SELECT id_artist, track_name FROM tracks', con=engine)
            # Creamos una marca de comparación para saber si el track ya existe para ese artista
            df_final['temp_check'] = df_final['id_artist'] + df_final['track_name']
            tracks_existentes['temp_check'] = tracks_existentes['id_artist'] + tracks_existentes['track_name']
            
            # Solo subimos los que no coinciden en esa combinación
            df_final = df_final[~df_final['temp_check'].isin(tracks_existentes['temp_check'])]
            df_final = df_final.drop(columns=['temp_check'])
        except:
            pass

        if not df_final.empty:
            df_final.to_sql('tracks', con=engine, if_exists='append', index=False)
            print(f"✅ {len(df_final)} tracks nuevos subidos desde {archivo}.")
        else:
            print(f"ℹ️ El archivo {archivo} no contenía canciones nuevas.")

In [106]:
# 1. Conectar
engine = conectar_db()

In [107]:
# 2. Subir Artistas y obtener el DF con IDs
df_maestro_artistas = subir_artistas('resultado_reggaeton.csv', engine)

✅ Se han subido 62 artistas nuevos.


In [108]:
# 2. Subir Artistas y obtener el DF con IDs
df_maestro_artistas = subir_artistas('resultado_latin.csv', engine)

✅ Se han subido 296 artistas nuevos.


In [109]:
# 2. Subir Artistas y obtener el DF con IDs
df_maestro_artistas = subir_artistas('resultado_indie.csv', engine)

✅ Se han subido 379 artistas nuevos.


In [110]:
df_maestro_artistas = subir_artistas('resultado_rock.csv', engine)


✅ Se han subido 495 artistas nuevos.


In [111]:
df_maestro_artistas = subir_artistas('resultado_hiphop.csv', engine)


✅ Se han subido 769 artistas nuevos.


In [112]:
# 3. Lista de tus archivos de géneros
archivos_a_subir = ['lista_hiphop.csv', 'lista_indie.csv', 'lista_latin.csv', 'lista_reggaeton.csv', 'lista_rock.csv']

In [113]:
# 4. Ejecutar subida de tracks
procesar_y_subir_tracks(archivos_a_subir, df_maestro_artistas, engine)

print("\n🚀 ¡Todo listo! Revisa tu Workbench.")

DatabaseError: Execution failed on sql 'INSERT INTO tracks (id_artist, id_genre, track_name, year) VALUES (:id_artist, :id_genre, :track_name, :year)': (mysql.connector.errors.IntegrityError) 1452 (23000): Cannot add or update a child row: a foreign key constraint fails (`music_db`.`tracks`, CONSTRAINT `fk_tracks_genre` FOREIGN KEY (`id_genre`) REFERENCES `genre` (`id_genre`))
[SQL: INSERT INTO tracks (id_artist, id_genre, track_name, year) VALUES (%(id_artist)s, %(id_genre)s, %(track_name)s, %(year)s)]
[parameters: [{'id_artist': '923b5d99-4020-40d8-9db5-1390986bb4cf', 'id_genre': 5, 'track_name': 'Afro Trap Pt. 3 (Champions League)', 'year': 2016}, {'id_artist': 'b1e26560-60e5-4236-bbdb-9aa5a8d5ee19', 'id_genre': 5, 'track_name': 'Congratulations', 'year': 2016}, {'id_artist': '6143403a-df6c-429e-8ee6-ef869896b0da', 'id_genre': 5, 'track_name': 'Congratulations', 'year': 2016}, {'id_artist': 'ab1d4693-41dd-44ed-9e9c-da2ff76f1a3a', 'id_genre': 5, 'track_name': 'Ganas', 'year': 2016}, 